
Accession Range Detector
========================
Scans XML corpus for accession number range patterns and identifies
documents requiring ground truth expansion.

AI6129 Project
Date: January 2026


In [9]:
import os
import re
import json
import logging
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, asdict
from datetime import datetime
from google.colab import drive


In [10]:
# ============================================================================
# Configuration
# ============================================================================
drive.mount('/content/drive')

# Declare configuration variables at the start
XML_DIRECTORY = "/content/drive/MyDrive/AI6129/xml"
OUTPUT_DIRECTORY = "/content/drive/MyDrive/AI6129/utility_output"
GT_DIRECTORY = "/content/drive/MyDrive/AI6129/ground_truth"
LOG_LEVEL = logging.INFO

# Range pattern indicators
RANGE_INDICATORS = [
    r'–',      # en-dash (Unicode U+2013)
    r'-',      # hyphen
    r'through',
    r'to',
]

# Valid two-letter prefixes for INSDC nucleotide accessions (direct submissions)
VALID_PREFIXES = [
    # NCBI
    'AF', 'AY', 'DQ', 'EF', 'EU', 'GQ', 'GU',
    'JF', 'JN', 'JQ', 'JX',
    'KC', 'KF', 'KJ', 'KM', 'KP', 'KR', 'KT', 'KU', 'KX', 'KY',
    'MF', 'MG', 'MH', 'MK', 'MN', 'MT', 'MW', 'MZ',
    'OK', 'OL', 'OM', 'ON', 'OP', 'OQ', 'OR',
    'PP', 'PQ', 'PV', 'PX',
    # EBI
    'AJ', 'AM',
    'FM', 'FN', 'FO', 'FP', 'FQ', 'FR',
    'HE', 'HF', 'HG', 'HH', 'HI',
    'LK', 'LL', 'LM', 'LN', 'LO', 'LR', 'LS', 'LT',
    'OA', 'OB', 'OC', 'OD', 'OE', 'OU', 'OV', 'OW', 'OX', 'OY', 'OZ',
    # DDBJ
    'AB', 'LC',
]

Mounted at /content/drive


In [11]:
# ============================================================================
# Data Classes
# ============================================================================

@dataclass
class RangeMatch:
    """Represents a detected accession range in text."""
    start_accession: str
    end_accession: str
    range_indicator: str
    matched_text: str
    context: str  # Surrounding text for verification
    expanded_count: int


@dataclass
class DocumentRangeReport:
    """Report for a single document's range detection results."""
    pmcid: str
    filepath: str
    ranges_found: List[RangeMatch]
    total_expanded_accessions: int
    needs_gt_update: bool

In [12]:
# ============================================================================
# Core Functions
# ============================================================================

def setup_logging(log_file: Optional[str] = None) -> logging.Logger:
    """
    Configure logging for the range detector.

    Args:
        log_file: Optional path to log file

    Returns:
        Configured logger instance
    """
    logger = logging.getLogger('RangeDetector')
    logger.setLevel(LOG_LEVEL)

    formatter = logging.Formatter(
        '%(asctime)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )

    # Console handler
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    # File handler (optional)
    if log_file:
        file_handler = logging.FileHandler(log_file)
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

    return logger


def build_range_pattern() -> re.Pattern:
    """
    Build compiled regex pattern for detecting accession ranges.

    Pattern matches structures like:
    - MH548440–MH548519 (en-dash)
    - MH548440-MH548519 (hyphen)
    - MW006474 through MW006482
    - AB123456 to AB123460

    Returns:
        Compiled regex pattern
    """
    # Build prefix alternation from valid prefixes
    prefix_pattern = '|'.join(VALID_PREFIXES)

    # Accession pattern: 2 letters + 5-6 digits (with optional version)
    accession_pattern = rf'({prefix_pattern})(\d{{5,6}})(?:\.\d+)?'

    # Range indicators
    range_sep = r'(?:\s*[–\-]\s*|\s+through\s+|\s+to\s+)'

    # Full range pattern
    full_pattern = rf'({accession_pattern}){range_sep}({accession_pattern})'

    return re.compile(full_pattern, re.IGNORECASE)


def extract_text_from_xml(filepath: str) -> str:
    """
    Extract text content from XML file.

    Args:
        filepath: Path to XML file

    Returns:
        Extracted text content
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()

        # Remove XML tags but preserve text
        text = re.sub(r'<[^>]+>', ' ', content)

        # Normalise whitespace
        text = re.sub(r'\s+', ' ', text)

        return text

    except Exception as e:
        raise IOError(f"Failed to read XML file {filepath}: {e}")


def extract_pmcid_from_filename(filename: str) -> str:
    """
    Extract PMCID from XML filename.

    Expected format: PMC{id}_{date}.xml or PMC{id}.xml

    Args:
        filename: XML filename

    Returns:
        PMCID string (e.g., 'PMC6935380')
    """
    match = re.match(r'(PMC\d+)', filename)
    if match:
        return match.group(1)
    return filename.replace('.xml', '')


def calculate_range_size(start_accession: str, end_accession: str) -> int:
    """
    Calculate the number of accessions in a range.

    Args:
        start_accession: Starting accession (e.g., 'MH548440')
        end_accession: Ending accession (e.g., 'MH548519')

    Returns:
        Number of accessions in range (inclusive)
    """
    # Extract numeric portions
    start_match = re.match(r'([A-Z]{2})(\d+)', start_accession.upper())
    end_match = re.match(r'([A-Z]{2})(\d+)', end_accession.upper())

    if not start_match or not end_match:
        return 0

    start_prefix = start_match.group(1)
    end_prefix = end_match.group(1)

    # Prefixes must match for valid range
    if start_prefix != end_prefix:
        return 0

    start_num = int(start_match.group(2))
    end_num = int(end_match.group(2))

    # End must be greater than or equal to start
    if end_num < start_num:
        return 0

    return end_num - start_num + 1


In [13]:
def get_context(text: str, match_start: int, match_end: int,
                context_chars: int = 50) -> str:
    """
    Extract surrounding context for a match.

    Args:
        text: Full text
        match_start: Start position of match
        match_end: End position of match
        context_chars: Number of characters to include before/after

    Returns:
        Context string with match highlighted
    """
    ctx_start = max(0, match_start - context_chars)
    ctx_end = min(len(text), match_end + context_chars)

    prefix = '...' if ctx_start > 0 else ''
    suffix = '...' if ctx_end < len(text) else ''

    return f"{prefix}{text[ctx_start:ctx_end]}{suffix}"


def detect_ranges_in_text(text: str, pattern: re.Pattern) -> List[RangeMatch]:
    """
    Detect all accession ranges in text.

    Args:
        text: Text content to search
        pattern: Compiled regex pattern

    Returns:
        List of RangeMatch objects
    """
    ranges = []

    for match in pattern.finditer(text):
        matched_text = match.group(0)

        # Extract start and end accessions
        # Groups: full_start, start_prefix, start_num, full_end, end_prefix, end_num
        start_accession = match.group(1)
        end_accession = match.group(4)

        # Determine range indicator used
        range_indicator = 'unknown'
        if '–' in matched_text:
            range_indicator = 'en-dash'
        elif '-' in matched_text:
            range_indicator = 'hyphen'
        elif 'through' in matched_text.lower():
            range_indicator = 'through'
        elif ' to ' in matched_text.lower():
            range_indicator = 'to'

        # Calculate expanded count
        expanded_count = calculate_range_size(start_accession, end_accession)

        # Skip if not a valid range (count <= 2 means just start and end)
        if expanded_count <= 2:
            continue

        # Get context
        context = get_context(text, match.start(), match.end())

        range_match = RangeMatch(
            start_accession=start_accession.upper(),
            end_accession=end_accession.upper(),
            range_indicator=range_indicator,
            matched_text=matched_text,
            context=context,
            expanded_count=expanded_count
        )

        ranges.append(range_match)

    return ranges


def scan_document(filepath: str, pattern: re.Pattern,
                  logger: logging.Logger) -> DocumentRangeReport:
    """
    Scan a single XML document for accession ranges.

    Args:
        filepath: Path to XML file
        pattern: Compiled regex pattern
        logger: Logger instance

    Returns:
        DocumentRangeReport with findings
    """
    filename = os.path.basename(filepath)
    pmcid = extract_pmcid_from_filename(filename)

    try:
        text = extract_text_from_xml(filepath)
        ranges = detect_ranges_in_text(text, pattern)

        total_expanded = sum(r.expanded_count for r in ranges)
        needs_update = len(ranges) > 0

        if needs_update:
            logger.info(
                f"{pmcid}: Found {len(ranges)} range(s), "
                f"{total_expanded} total accessions"
            )

        return DocumentRangeReport(
            pmcid=pmcid,
            filepath=filepath,
            ranges_found=ranges,
            total_expanded_accessions=total_expanded,
            needs_gt_update=needs_update
        )

    except Exception as e:
        logger.error(f"Error processing {pmcid}: {e}")
        return DocumentRangeReport(
            pmcid=pmcid,
            filepath=filepath,
            ranges_found=[],
            total_expanded_accessions=0,
            needs_gt_update=False
        )


def scan_corpus(xml_directory: str, logger: logging.Logger) -> List[DocumentRangeReport]:
    """
    Scan entire XML corpus for accession ranges.

    Args:
        xml_directory: Directory containing XML files
        logger: Logger instance

    Returns:
        List of DocumentRangeReport objects
    """
    pattern = build_range_pattern()
    reports = []

    xml_files = list(Path(xml_directory).glob('*.xml'))
    total_files = len(xml_files)

    logger.info(f"Scanning {total_files} XML files for accession ranges...")

    for idx, filepath in enumerate(xml_files, 1):
        if idx % 50 == 0:
            logger.info(f"Progress: {idx}/{total_files} files processed")

        report = scan_document(str(filepath), pattern, logger)
        reports.append(report)

    return reports


In [14]:
def generate_summary_report(reports: List[DocumentRangeReport],
                           output_path: str,
                           logger: logging.Logger) -> Dict:
    """
    Generate summary report of range detection results.

    Args:
        reports: List of document reports
        output_path: Path for output files
        logger: Logger instance

    Returns:
        Summary statistics dictionary
    """
    # Filter documents with ranges
    docs_with_ranges = [r for r in reports if r.needs_gt_update]

    summary = {
        'scan_timestamp': datetime.now().isoformat(),
        'total_documents_scanned': len(reports),
        'documents_with_ranges': len(docs_with_ranges),
        'total_ranges_found': sum(len(r.ranges_found) for r in docs_with_ranges),
        'total_expanded_accessions': sum(r.total_expanded_accessions for r in docs_with_ranges),
        'documents_requiring_gt_update': [r.pmcid for r in docs_with_ranges]
    }

    # Write summary JSON
    summary_file = os.path.join(output_path, 'range_detection_summary.json')
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    # Write detailed report for documents with ranges
    detailed_file = os.path.join(output_path, 'range_detection_detailed.json')
    detailed_data = []

    for report in docs_with_ranges:
        doc_data = {
            'pmcid': report.pmcid,
            'filepath': report.filepath,
            'total_expanded_accessions': report.total_expanded_accessions,
            'ranges': [asdict(r) for r in report.ranges_found]
        }
        detailed_data.append(doc_data)

    with open(detailed_file, 'w', encoding='utf-8') as f:
        json.dump(detailed_data, f, indent=2)

    # Write CSV for easy review
    csv_file = os.path.join(output_path, 'range_detection_inventory.csv')
    with open(csv_file, 'w', encoding='utf-8') as f:
        f.write('pmcid,start_accession,end_accession,expanded_count,range_indicator,matched_text\n')
        for report in docs_with_ranges:
            for rng in report.ranges_found:
                # Escape matched_text for CSV
                matched_escaped = rng.matched_text.replace('"', '""')
                f.write(
                    f'{report.pmcid},{rng.start_accession},{rng.end_accession},'
                    f'{rng.expanded_count},{rng.range_indicator},"{matched_escaped}"\n'
                )

    logger.info(f"Summary written to: {summary_file}")
    logger.info(f"Detailed report written to: {detailed_file}")
    logger.info(f"CSV inventory written to: {csv_file}")

    return summary


def print_summary(summary: Dict) -> None:
    """
    Print summary statistics to console.

    Args:
        summary: Summary statistics dictionary
    """
    print("\n" + "=" * 70)
    print("ACCESSION RANGE DETECTION SUMMARY")
    print("=" * 70)
    print(f"Scan timestamp:              {summary['scan_timestamp']}")
    print(f"Total documents scanned:     {summary['total_documents_scanned']}")
    print(f"Documents with ranges:       {summary['documents_with_ranges']}")
    print(f"Total ranges found:          {summary['total_ranges_found']}")
    print(f"Total expanded accessions:   {summary['total_expanded_accessions']}")
    print("-" * 70)

    if summary['documents_requiring_gt_update']:
        print("\nDocuments requiring GT update:")
        for pmcid in summary['documents_requiring_gt_update']:
            print(f"  - {pmcid}")
    else:
        print("\nNo documents with accession ranges detected.")

    print("=" * 70 + "\n")

In [15]:
# ============================================================================
# Main Entry Point
# ============================================================================

def main(xml_directory: str, output_directory: str,
         log_file: Optional[str] = None) -> Dict:
    """
    Main entry point for range detection.

    Args:
        xml_directory: Directory containing XML files to scan
        output_directory: Directory for output files
        log_file: Optional path to log file

    Returns:
        Summary statistics dictionary
    """
    # Create output directory if needed
    os.makedirs(output_directory, exist_ok=True)

    # Setup logging
    logger = setup_logging(log_file)
    logger.info("Starting accession range detection...")

    # Scan corpus
    reports = scan_corpus(xml_directory, logger)

    # Generate reports
    summary = generate_summary_report(reports, output_directory, logger)

    # Print summary
    print_summary(summary)

    logger.info("Range detection complete.")

    return summary

In [17]:
summary = main(
    xml_directory=XML_DIRECTORY,
    output_directory= OUTPUT_DIRECTORY,
    log_file= OUTPUT_DIRECTORY + "accession_range.log"  # Set to a path if you want file logging
)

2026-01-21 07:37:33 - INFO - Starting accession range detection...
2026-01-21 07:37:33 - INFO - Starting accession range detection...
INFO:RangeDetector:Starting accession range detection...
2026-01-21 07:37:33 - INFO - Scanning 227 XML files for accession ranges...
2026-01-21 07:37:33 - INFO - Scanning 227 XML files for accession ranges...
INFO:RangeDetector:Scanning 227 XML files for accession ranges...
2026-01-21 07:37:37 - INFO - Progress: 50/227 files processed
2026-01-21 07:37:37 - INFO - Progress: 50/227 files processed
INFO:RangeDetector:Progress: 50/227 files processed
2026-01-21 07:37:40 - INFO - Progress: 100/227 files processed
2026-01-21 07:37:40 - INFO - Progress: 100/227 files processed
INFO:RangeDetector:Progress: 100/227 files processed
2026-01-21 07:37:40 - INFO - PMC7441611: Found 1 range(s), 63 total accessions
2026-01-21 07:37:40 - INFO - PMC7441611: Found 1 range(s), 63 total accessions
INFO:RangeDetector:PMC7441611: Found 1 range(s), 63 total accessions
2026-01-2


ACCESSION RANGE DETECTION SUMMARY
Scan timestamp:              2026-01-21T07:37:48.997870
Total documents scanned:     227
Documents with ranges:       3
Total ranges found:          6
Total expanded accessions:   108
----------------------------------------------------------------------

Documents requiring GT update:
  - PMC7441611
  - PMC7557518
  - PMC7764154



In [ ]:
if __name__ == '__main__':
    import argparse

    parser = argparse.ArgumentParser(
        description='Detect accession number ranges in XML corpus'
    )
    parser.add_argument(
        'xml_directory',
        help='Directory containing XML files to scan'
    )
    parser.add_argument(
        'output_directory',
        help='Directory for output files'
    )
    parser.add_argument(
        '--log-file',
        help='Optional path to log file',
        default=None
    )

    args = parser.parse_args()

    main(args.xml_directory, args.output_directory, args.log_file)
